In [0]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import VectorAssembler

# 1. Prepare features into a single vector column
assembler = VectorAssembler(
    inputCols=['total_time_consumed', 'sleep_hours_per_night', 'exercise_hours_per_week', 'daily_steps_count'],
    outputCol="features")

df_spark = spark.table("instagram.goldlayer.vw_health_behavior_analysis")
data = assembler.transform(df_spark)

# 2. Train using Spark's distributed RF
rf = RandomForestRegressor(featuresCol="features", labelCol="user_engagement_score", numTrees=100)
model = rf.fit(data)

# 3. Predict and Save
predictions = model.transform(data)
predictions.select("user_id", "prediction", "total_time_consumed", "sleep_hours_per_night", "exercise_hours_per_week", "daily_steps_count") \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("instagram.model_output.ml_engagement")